In [1]:
from pathlib import Path
import re


def find_repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'README.md').is_file() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('Could not find the repository root.')


ROOT = find_repository_root()
DATA_DIR = ROOT / 'data'

STATES = ('healthy', 'front_ball', 'rear_ball', 'misalignment', 'demag')
QUALITIES = (('Clean', 'clean'), ('ENV', 'ENV'), ('SF', 'SF'))
SOURCES = ('mait', 'bat')
STATUS = {state: '100%' for state in STATES}
STATUS['demag'] = 'WIP'
DESCRIPTIONS = {
    'healthy': 'Reference operating condition without the faults.',
    'front_ball': 'Fault condition associated with the front bearing.',
    'rear_ball': 'Fault condition associated with the rear bearing.',
    'misalignment': 'Fault condition associated with shaft misalignment.',
    'demag': 'Fault condition associated with weakened motor magnets.',
}
FILENAME = re.compile(
    r'^analize_(?P<state>healthy|front_ball|rear_ball|misalignment|demag)'
    r'(?P<id>\d+)(?:_(?P<quality>ENV|SF))?_'
    r'\d+rpm_\d+mA_(?P<source>mait|bat)\.csv$'
)


def format_size(size_bytes):
    value = float(size_bytes)
    for unit in ('B', 'kB', 'MB', 'GB', 'TB'):
        if value < 1000 or unit == 'TB':
            if unit == 'B':
                return f'{int(value)} B'
            return f"{value:.2f}".rstrip('0').rstrip('.') + f' {unit}'
        value /= 1000


totals = {
    state: {quality: {source: [0, 0] for source in SOURCES} for _, quality in QUALITIES}
    for state in STATES
}

for path in DATA_DIR.rglob('analize_*.csv'):
    match = FILENAME.match(path.name)
    if match is None:
        continue
    state = match['state']
    quality = match['quality'] or 'clean'
    source = match['source']
    totals[state][quality][source][0] += 1
    totals[state][quality][source][1] += path.stat().st_size


def descriptor(state):
    groups = []
    for label, quality in QUALITIES:
        entries = []
        for source in SOURCES:
            count, size_bytes = totals[state][quality][source]
            noun = 'file' if count == 1 else 'files'
            entries.append(f'`{source}` {count} CSV {noun}, {format_size(size_bytes)}')
        groups.append(f'{label}: ' + '; '.join(entries))
    return '<br>'.join(groups)


lines = [
    '| State | Status | Data descriptors | Description |',
    '| --- | --- | --- | --- |',
]
for state in STATES:
    lines.append(f'| {state} | {STATUS[state]} | {descriptor(state)} | {DESCRIPTIONS[state]} |')

summary_table = '\n'.join(lines)
print(summary_table)


| State | Status | Data descriptors | Description |
| --- | --- | --- | --- |
| healthy | 100% | Clean: `mait` 70 CSV files, 1.22 GB; `bat` 28 CSV files, 618.51 MB<br>ENV: `mait` 1 CSV file, 46.83 MB; `bat` 5 CSV files, 164.39 MB<br>SF: `mait` 0 CSV files, 0 B; `bat` 0 CSV files, 0 B | Reference operating condition without the faults. |
| front_ball | 100% | Clean: `mait` 63 CSV files, 433.35 MB; `bat` 0 CSV files, 0 B<br>ENV: `mait` 1 CSV file, 7.28 MB; `bat` 0 CSV files, 0 B<br>SF: `mait` 1 CSV file, 8.28 MB; `bat` 0 CSV files, 0 B | Fault condition associated with the front bearing. |
| rear_ball | 100% | Clean: `mait` 63 CSV files, 437.34 MB; `bat` 6 CSV files, 47.22 MB<br>ENV: `mait` 0 CSV files, 0 B; `bat` 0 CSV files, 0 B<br>SF: `mait` 2 CSV files, 11.47 MB; `bat` 0 CSV files, 0 B | Fault condition associated with the rear bearing. |
| misalignment | 100% | Clean: `mait` 63 CSV files, 432.53 MB; `bat` 9 CSV files, 138.8 MB<br>ENV: `mait` 3 CSV files, 22.01 MB; `bat` 0 CSV files,